# SPIN-Soft — Step 2/4: HSIC (decoupling) + capability (perplexity **+ MMLU**) + **HSIC-drop bootstrap CIs**, swept over α

This is the core decoupling result for Contribution 1. It loads the Step-1 coupled set
(`coupled_soft_<tag>.json`), loads the model **once**, caches the original coupled-weight values
**once**, then sweeps the α grid *internally*: for each α it resets the coupled weights to their
originals, rescales them to **αW**, and measures fairness↔privacy dependence (HSIC, frozen-σ grid),
WikiText-2 perplexity, **and MMLU('all') accuracy**. No generation, no judge — forward passes only.

**Why an internal loop, not one run per α:** soft-scaling is *not* idempotent ($W\to\alpha W$ twice
gives $\alpha^2 W$), so each α must start from the cached originals — but once cached, the whole grid
runs in a single session with the model loaded once. α=0 reproduces hard SPIN; α=1 reproduces baseline.

**Prediction under test (Contribution 1):** $I(F_\alpha;P_\alpha)$ — hence HSIC — is **monotonically
increasing in α**; capability (perplexity, MMLU) is best near α=1 and worst near α=0, with the soft
regime trading between.

**New metrics (parity with Iter-SPIN / healthcare Step 2):**
- **MMLU accuracy** per α (seeded `N_MMLU=1000` subsample of `cais/mmlu:all`, length-normalised
  choice log-likelihood) — the capability/utility-preservation check that was missing here.
- **Bootstrap 95% CIs** (`BOOT=500`, seed 0) on the *percent HSIC drop vs baseline* at each α — the
  significance test for decoupling. Written as `bootstrap.drop_ci` (Step 4 cell).

**Input:** add this model's `coupled_soft_<tag>.json` (Step-1 output) via *Add Input*.
**Output:** `metrics_soft_<tag>.json` (now carries `n_mmlu`, per-α `mmlu`, and `bootstrap.drop_ci`).
Checkpointed per α (`metrics_ckpt_<tag>.json`) → resumes on timeout; an α missing `mmlu` is recomputed.
GPU **T4 x2**, Internet ON. Forward-only; MMLU adds ~N_MMLU×#choices passes per α, so budget more time
(whole grid ≈ tens of minutes POC / ~3–4 h for 7B — drop N_MMLU to trade accuracy fidelity for speed).


## Step 0 — env + config + **model select** + find coupled set

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, re, gc, time, json
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ============================================================
# SELECT MODEL — uncomment EXACTLY ONE (MODEL_ID + tokenizer flag together).
# ------------------------------------------------------------
# MODEL_ID,USE_FAST = 'google/gemma-2-2b-it',            False   # POC (gated)
MODEL_ID,USE_FAST = 'meta-llama/Llama-3.2-3B-Instruct', True   # POC (gated; fast tokenizer)
# MODEL_ID,USE_FAST = 'Qwen/Qwen2.5-3B-Instruct',        False   # POC
# MODEL_ID,USE_FAST = 'Qwen/Qwen2-7B-Instruct',          False   # scale-up
# ============================================================
# --- alpha grid (locked): 0 = hard SPIN, 1 = baseline ---
ALPHAS=[0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
# 7B tip: drop 0.1 to save one pass; 1.0 duplicates the explicit baseline below
#         (kept as a reset-logic sanity check — safe to remove on the 7B).

HSIC_N=300
PPL_SEQLEN=2048
SIGMAS=[50,100,200,400]     # authors' Gaussian-kernel sigma grid (Appendix F)
USE_CHAT_TEMPLATE=False      # authors' raw-question protocol
DTYPE=torch.float16
# --- new metrics (parity with Iter-SPIN / healthcare Step 2) ---
N_MMLU=1000                  # seeded MMLU('all') subsample -> capability accuracy per alpha (0 = skip MMLU)
SEED=0                       # MMLU subsample seed (matches Iter-SPIN Step 2)
BOOT=500                     # bootstrap resamples for HSIC-drop 95% CIs (0 = skip)
MODEL_TAG=MODEL_ID.split('/')[-1]

def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
CP=find(f'coupled_soft_{MODEL_TAG}.json'); assert CP, f'add coupled_soft_{MODEL_TAG}.json (Step-1 output) as input'
meta=json.load(open(CP)); COUPLED=meta['coupled']
NZ=sum(len(v) for v in COUPLED.values())
print('SoftSPIN Step 2 |', MODEL_TAG, '| coupled set:', CP, '| matrices', len(COUPLED), '| coupled weights', NZ)
print('alpha grid:', ALPHAS, '| sigmas', SIGMAS, '| N_MMLU', N_MMLU, '| BOOT', BOOT)


## Step 0b — Hugging Face auth (gemma-2 & Llama-3.2 are gated)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    _tok=UserSecretsClient().get_secret('HF')
    from huggingface_hub import login; login(token=_tok); print('HF login OK (Kaggle secret HF)')
except Exception as e:
    print('No Kaggle HF secret picked up:', repr(e), '-> fine for Qwen (open).')

## Step 1 — data (SALAD reps + WikiText-2) + model + **cache original coupled weights**

In [ ]:
DEAN_DIR='/tmp/DEAN'   # clone to scratch, NOT /kaggle/working, so DEAN is never saved to notebook output
if not os.path.exists(DEAN_DIR):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git',DEAN_DIR], check=True)
DATA=f'{DEAN_DIR}/src/data'
fair_q=pd.read_csv(f'{DATA}/salad_fairness.csv')['questions'].tolist()[:HSIC_N]
priv_q=pd.read_csv(f'{DATA}/salad_privacy.csv')['questions'].tolist()[:HSIC_N]
from datasets import load_dataset
_wt=load_dataset('wikitext','wikitext-2-raw-v1',split='test')['text']

# ---- MMLU (all subjects), seeded subsample -> capability accuracy (parity with Iter-SPIN Step 2) ----
mmlu=[]
if N_MMLU:
    try: _mm=load_dataset('cais/mmlu','all',split='test')
    except Exception: _mm=load_dataset('cais/mmlu','all',split='test',trust_remote_code=True)
    _mm=_mm.shuffle(seed=SEED).select(range(min(N_MMLU,len(_mm))))
    mmlu=[{'prompt':f"Question: {r['question']}\nAnswer:",'choices':[f" {c}" for c in r['choices']],'answer':int(r['answer'])} for r in _mm]

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()  # cap sized for 7B; harmless for POC
model.config.use_cache=False   # forward-only passes; avoids KV-cache cross-device writes on multi-GPU shards
GEN_DEV=model.get_input_embeddings().weight.device
wt_ids=tokenizer('\n\n'.join(_wt), return_tensors='pt').input_ids
print('model loaded | entry', GEN_DEV, '|', len(fair_q), 'fair +', len(priv_q), 'priv reps | MMLU', len(mmlu))

# ---- cache the ORIGINAL coupled-weight values ONCE (weights still untouched = baseline) ----
PARAMS=dict(model.named_parameters()); IDX={}; ORIG={}
for n,idxlist in COUPLED.items():
    if n in PARAMS and idxlist:
        p=PARAMS[n]; idx=torch.tensor(idxlist, device=p.device)
        IDX[n]=idx; ORIG[n]=p.data.view(-1)[idx].clone()   # on-device snapshot
print('cached originals for', len(IDX), 'matrices,', sum(v.numel() for v in IDX.values()), 'weights')


## Step 2 — functions (raw reps, frozen-σ HSIC, perplexity, **MMLU accuracy**, reset+scale-to-αW)

`reps`, `hsic`, `eval_ppl` are verbatim from the 7B Step 2. `_choice_loglik`/`mc_accuracy` are the
length-normalised multiple-choice scorer ported verbatim from Iter-SPIN Step 2. `apply_alpha`
**absolutely assigns** each coupled weight to `α · W_original` (so repeated α values never compound).
`measure()` now returns HSIC, perplexity, MMLU accuracy **and the last-layer reps** — the reps are
kept so the Step-4 cell can bootstrap the HSIC drop without a second forward pass.


In [ ]:
def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
@torch.no_grad()
def reps(queries,bs=4):                         # last-layer mean-pooled reps (used by the bootstrap backfill)
    out=[]
    for i in range(0,len(queries),bs):
        enc=tokenizer(fmt(queries[i:i+bs]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        h=model(**enc,output_hidden_states=True).hidden_states[-1]; m=enc.attention_mask.unsqueeze(-1)
        out.append(((h*m).sum(1)/m.sum(1)).float().cpu().numpy())
        print(f'    reps {min(i+bs,len(queries))}/{len(queries)}',end='\r')
    print(); return np.concatenate(out,0)
@torch.no_grad()
def reps_all(queries,bs=4):                     # per-layer mean-pooled reps (ALL hidden layers) -> per-layer HSIC
    acc=None
    for i in range(0,len(queries),bs):
        enc=tokenizer(fmt(queries[i:i+bs]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        hs=model(**enc,output_hidden_states=True).hidden_states; m=enc.attention_mask.unsqueeze(-1)
        pooled=[((h*m).sum(1)/m.sum(1)).float().cpu().numpy() for h in hs]
        acc=[[p] for p in pooled] if acc is None else [acc[l]+[pooled[l]] for l in range(len(pooled))]
        print(f'    reps {min(i+bs,len(queries))}/{len(queries)}',end='\r')
    print(); return [np.concatenate(a,0) for a in acc]
def _d2(A,B):
    sa=(A**2).sum(1,keepdims=True); sb=(B**2).sum(1,keepdims=True); return np.clip(sa+sb.T-2*A@B.T,0,None)
def hsic(X,Y,sigma):
    # authors' HSIC: tr(K H L H)/(n-1)^2, Gaussian kernel exp(-||x-y||^2/(2 sigma^2))
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]
    K=np.exp(-_d2(X,X)/(2*sigma**2)); L=np.exp(-_d2(Y,Y)/(2*sigma**2)); H=np.eye(n)-np.ones((n,n))/n
    return float(np.trace(K@H@L@H)/(n-1)**2)
def nhsic(X,Y,sigma):                           # normalised HSIC in [0,1] (norm-robust) — parity with Iter/health
    hxy=hsic(X,Y,sigma); hxx=hsic(X,X,sigma); hyy=hsic(Y,Y,sigma)
    return float(hxy/np.sqrt(hxx*hyy)) if hxx>0 and hyy>0 else 0.0
def linear_cka(X,Y):                            # linear CKA, scale-invariant
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]       # match HSIC's min-truncation (fair/priv sizes may differ)
    Xc=X-X.mean(0,keepdims=True); Yc=Y-Y.mean(0,keepdims=True)
    num=np.linalg.norm(Yc.T@Xc)**2; den=np.linalg.norm(Xc.T@Xc)*np.linalg.norm(Yc.T@Yc)
    return float(num/den) if den>0 else 0.0
def mean_norm(X): return float(np.linalg.norm(X,axis=1).mean())
def full_pack(Lf,Lp):
    # Lf,Lp = per-layer rep lists; final layer carries the headline stats
    # (parity with Iter-SPIN / healthcare Step 2: HSIC + nHSIC + CKA + norms + per-layer HSIC)
    Xf,Xp=Lf[-1],Lp[-1]
    return {'hsic':{str(s):hsic(Xf,Xp,s) for s in SIGMAS},
            'hsic_per_layer':{str(s):[hsic(Lf[l],Lp[l],s) for l in range(len(Lf))] for s in SIGMAS},
            'nhsic':{str(s):nhsic(Xf,Xp,s) for s in SIGMAS},
            'cka':linear_cka(Xf,Xp),
            'norm_fair':mean_norm(Xf),'norm_priv':mean_norm(Xp)}
@torch.no_grad()
def eval_ppl(seqlen=PPL_SEQLEN):
    model.config.use_cache=False; ns=wt_ids.numel()//seqlen; nlls=[]
    for i in range(ns):
        inp=wt_ids[:,i*seqlen:(i+1)*seqlen].to(GEN_DEV); lg=model(inp).logits
        loss=torch.nn.CrossEntropyLoss()(lg[:,:-1,:].reshape(-1,lg.size(-1)),inp[:,1:].reshape(-1))
        nlls.append(loss.float()*seqlen)
        print(f'    ppl {i+1}/{ns}',end='\r')
    print(); return torch.exp(torch.stack(nlls).sum()/(ns*seqlen)).item()
# ---- capability accuracy via length-normalised choice log-likelihood (verbatim from Iter-SPIN Step 2) ----
@torch.no_grad()
def _choice_loglik(prompt, choice):
    enc=tokenizer(prompt+choice, return_tensors='pt', truncation=True, max_length=1024).to(GEN_DEV)
    plen=tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
    logits=model(**enc,use_cache=False).logits[:, :-1, :]; tgt=enc.input_ids[:, 1:]
    logp=torch.log_softmax(logits, dim=-1).gather(2, tgt.unsqueeze(-1)).squeeze(-1)[0]
    seg=logp[plen-1:]
    return float(seg.mean().item()) if seg.numel()>0 else -1e9
def mc_accuracy(items):
    if not items: return float('nan')
    c=0
    for j,it in enumerate(items):
        lls=[_choice_loglik(it['prompt'], ch) for ch in it['choices']]
        c+= int(np.argmax(lls))==it['answer']
        print(f'    mmlu {j+1}/{len(items)}',end='\r')
    print(); return c/len(items)
@torch.no_grad()
def apply_alpha(alpha):
    # absolute assignment: coupled weight <- alpha * original  (idempotent across alpha values)
    nz=0
    for n,idx in IDX.items():
        PARAMS[n].data.view(-1)[idx]=alpha*ORIG[n]; nz+=idx.numel()
    return nz
def measure():
    # full decoupling pack (HSIC/nHSIC/CKA/norms + per-layer HSIC), perplexity, MMLU accuracy,
    # and last-layer reps (kept for the HSIC-drop bootstrap)
    Lf,Lp=reps_all(fair_q),reps_all(priv_q)
    fp=full_pack(Lf,Lp)
    return fp, eval_ppl(), mc_accuracy(mmlu), Lf[-1], Lp[-1]
print('functions ready.')


## Step 3 — explicit baseline, then sweep α (checkpointed, resumable) — now with MMLU

In [ ]:
CK=f'/kaggle/working/metrics_ckpt_{MODEL_TAG}.json'
state=json.load(open(CK)) if os.path.exists(CK) else {'baseline':None,'per_alpha':{}}
REPS={}   # last-layer reps for the HSIC-drop bootstrap (this session; backfilled below if resumed)

# --- explicit baseline (weights untouched); recompute if an old ckpt lacks the full pack (nhsic) ---
if state['baseline'] is None or 'nhsic' not in state['baseline']:
    print('baseline (unmodified weights)...')
    fp,b_p,b_acc,Xf0,Xp0=measure()
    state['baseline']={**fp,'ppl':b_p,'mmlu':b_acc}; json.dump(state,open(CK,'w'))
    REPS['base']=(Xf0,Xp0)
print('baseline HSIC', state['baseline']['hsic'], '| ppl', round(state['baseline']['ppl'],3),
      '| MMLU', round(state['baseline'].get('mmlu',float('nan')),4))

# --- alpha sweep (an alpha missing the full pack from an older run is recomputed) ---
for a in ALPHAS:
    key=f'{a:.2f}'
    if key in state['per_alpha'] and 'nhsic' in state['per_alpha'][key]:
        print(f'alpha={key} already done, skipping'); continue
    t0=time.time(); nz=apply_alpha(a)
    print(f'alpha={key}: set {nz} coupled weights to alpha*W ; measuring...')
    fp,pp,acc,Xf,Xp=measure()
    state['per_alpha'][key]={'alpha':a,**fp,'ppl':pp,'mmlu':acc,'coupled_scaled':int(nz)}
    json.dump(state,open(CK,'w')); REPS[key]=(Xf,Xp)
    print(f'  alpha={key} done ({time.time()-t0:.0f}s) | ppl {pp:.3f} | MMLU {acc:.4f}')

res={'model':MODEL_ID,'tag':MODEL_TAG,'variant':'soft-SPIN','p':meta.get('p'),'q':meta.get('q'),
     'coupled_count':NZ,'sigmas':SIGMAS,'alphas':ALPHAS,'n_mmlu':len(mmlu),
     'baseline':state['baseline'],'per_alpha':state['per_alpha']}
OUT=f'/kaggle/working/metrics_soft_{MODEL_TAG}.json'
json.dump(res, open(OUT,'w'), indent=2)

# --- report 1: raw HSIC (per sigma) + capability ---
print('\n'+'='*90)
print(f'  SPIN-Soft Step 2 — {MODEL_TAG} | HSIC (fairness<->privacy) vs alpha, per sigma')
print('='*90)
hdr=f'{"alpha":>7}' + ''.join(f'{"s="+str(s):>13}' for s in SIGMAS) + f'{"ppl":>11}' + f'{"MMLU":>9}'
print(hdr); print('-'*90)
b=state['baseline']
print(f'{"base":>7}' + ''.join(f'{b["hsic"][str(s)]:>13.6f}' for s in SIGMAS) + f'{b["ppl"]:>11.3f}' + f'{b.get("mmlu",float("nan")):>9.4f}')
for a in ALPHAS:
    e=state['per_alpha'][f'{a:.2f}']
    print(f'{a:>7.2f}' + ''.join(f'{e["hsic"][str(s)]:>13.6f}' for s in SIGMAS) + f'{e["ppl"]:>11.3f}' + f'{e.get("mmlu",float("nan")):>9.4f}')
print('-'*90)

# --- report 2: norm-robust decoupling (nHSIC) + linear CKA + representation norms ---
print(f'{"alpha":>7} {"nHSIC@50":>10} {"nHSIC@200":>11} {"CKA":>9} {"norm_fair":>11} {"norm_priv":>11}')
print('-'*66)
for tag_,e in [('base',b)]+[(f'{a:.2f}',state['per_alpha'][f'{a:.2f}']) for a in ALPHAS]:
    print(f'{tag_:>7} {e["nhsic"]["50"]:>10.4f} {e["nhsic"]["200"]:>11.4f} {e["cka"]:>9.4f} {e["norm_fair"]:>11.3f} {e["norm_priv"]:>11.3f}')
print('-'*66)
print('Full pack stored per alpha: hsic, hsic_per_layer, nhsic, cka, norm_fair, norm_priv (+ ppl, mmlu).')
print('Expect (Contribution 1): HSIC/nHSIC rise with alpha (0 -> baseline); CKA tracks; ppl lowest at alpha=1; MMLU ~preserved.')
print('saved', OUT, '(bootstrap CIs added by the next cell)')


## Step 4 — HSIC-drop bootstrap 95% CIs (per α, per σ)

Resamples the paired fairness/privacy representations (`BOOT=500`, seed 0) to put a 95% CI on the
**percent change in HSIC vs baseline** at each α — same centred-kernel estimator and resampling
scheme as Iter-SPIN / healthcare Step 2. Reuses the last-layer reps cached during the sweep and
recomputes reps **only** for α's missing after a checkpoint resume (reps are cheap; HSIC/PPL/MMLU
stay checkpointed). Adds `bootstrap.drop_ci` (keyed `a<α>_s<σ>`) to `metrics_soft_<tag>.json`.
A CI lying strictly below 0 (small α) = the decoupling at that α is significant.


In [ ]:
# --- ensure last-layer reps exist for baseline + every alpha (backfill only if resumed from ckpt) ---
if BOOT:
    if 'base' not in REPS:
        apply_alpha(1.0)                       # 1.0*W_orig == untouched baseline weights
        REPS['base']=(reps(fair_q),reps(priv_q))
    for a in ALPHAS:
        key=f'{a:.2f}'
        if key not in REPS:
            apply_alpha(a); REPS[key]=(reps(fair_q),reps(priv_q))

if BOOT and REPS:
    Xf0,Xp0=REPS['base']
    n0=min(len(Xf0),len(Xp0)); rng=np.random.default_rng(0)
    IDXB=[rng.integers(0,n0,n0) for _ in range(BOOT)]
    def _kern(X,sig): return np.exp(-_d2(X[:n0],X[:n0])/(2*sig**2))
    def _hsic_idx(Kf,Lp,idx):
        Kb=Kf[np.ix_(idx,idx)]; Lb=Lp[np.ix_(idx,idx)]; n=len(idx)
        Kc=Kb-Kb.mean(0,keepdims=True)-Kb.mean(1,keepdims=True)+Kb.mean()
        return (Kc*Lb).sum()/(n-1)**2
    BK={s:(_kern(Xf0,s),_kern(Xp0,s)) for s in SIGMAS}   # baseline kernels cached once
    ci={}
    for a in ALPHAS:
        key=f'{a:.2f}'; Xf,Xp=REPS[key]
        for s in SIGMAS:
            Kf,Lp=_kern(Xf,s),_kern(Xp,s); Kf0,Lp0=BK[s]; arr=np.empty(BOOT)
            for bi,idx in enumerate(IDXB):
                base=_hsic_idx(Kf0,Lp0,idx); st=_hsic_idx(Kf,Lp,idx)
                arr[bi]=100*(st-base)/base if base else 0.0
            ci[f'a{key}_s{s}']={'med':float(np.median(arr)),'lo':float(np.percentile(arr,2.5)),'hi':float(np.percentile(arr,97.5))}
        print(f'  bootstrap alpha={key} done')
    res['bootstrap']={'B':BOOT,'drop_ci':ci}
    json.dump(res, open(OUT,'w'), indent=2)
    print('\nbootstrap HSIC-drop CIs added (B=%d) -> %s'%(BOOT,OUT))

    # --- report: HSIC drop% vs baseline with 95% CI (sigma=50) ---
    print('\n'+'-'*58); print(f'{"alpha":>7} {"dHSIC@50 med%":>15} {"lo95":>10} {"hi95":>10}'); print('-'*58)
    for a in ALPHAS:
        e=ci[f'a{a:.2f}_s50']
        print(f'{a:>7.2f} {e["med"]:>+14.1f} {e["lo"]:>+10.1f} {e["hi"]:>+10.1f}')
else:
    print('BOOT=0 or no reps: bootstrap skipped.')
